# NLP Lab 3: Word Embeddings and Foundational Classifiers
Welcome to NLP Lab 3! Having explored Bag of Words, TF-IDF and N-Grams, we now shift our focus to more advanced text representations and supervised machine learning. 

In this lab, you will build three fundamental algorithms completely from scratch using `numpy`:
1. **Word2Vec (Skip-gram):** Learning dense continuous word embeddings.
2. **Naive Bayes Classifier:** A generative probabilistic model for text classification.
3. **Logistic Regression:** A discriminative linear classifier.

Let's import our mathematical foundation: `numpy`.

In [9]:
import numpy as np
import re
from collections import defaultdict, Counter

# Set random seed for reproducibility
np.random.seed(42)

---
## Topic 1: Word2Vec (Skip-Gram Architecture) From Scratch

Unlike Bag of Words, Word2Vec learns **dense embeddings**—vectors where words with similar meanings are located close to each other in vector space. 

In the **Skip-gram** model, we use a center word to predict its surrounding context words. The architecture is a shallow neural network with one hidden layer.

**The Math:**
Given a center word $w_c$, we predict a context word $w_o$. We pass the input through a hidden weight matrix $W_1$ (our embedding matrix) and an output weight matrix $W_2$. We then apply the **Softmax** function to get a probability distribution over the vocabulary:

$$\hat{y}_i = \frac{e^{z_i}}{\sum_{j=1}^{|V|} e^{z_j}}$$

Our goal is to minimize the Cross-Entropy Loss using Gradient Descent.

In [10]:
# 1. Toy Corpus and Preprocessing
corpus = "the king is a strong man and the queen is a wise woman"
tokens = corpus.split()

vocab = list(set(tokens))
word2id = {w: i for i, w in enumerate(vocab)}
id2word = {i: w for i, w in enumerate(vocab)}
V = len(vocab)

print(f"Vocabulary Size: {V}")

# 2. Generate Skip-gram Training Data (Window Size = 1)
window_size = 1
training_data = []

for i in range(len(tokens)):
    center_word = tokens[i]
    # Get context words within the window
    context_words = tokens[max(0, i - window_size) : i] + tokens[i + 1 : min(len(tokens), i + window_size + 1)]
    print(f"Center Word: {center_word}, Context Words: {context_words}")
    
    for context in context_words:
        training_data.append((word2id[center_word], word2id[context]))
        
    print(f"Training Data: {training_data}")
    
def one_hot(word_id, vocab_size):
    vec = np.zeros(vocab_size)
    vec[word_id] = 1.0
    return vec

def softmax(x):
    e_x = np.exp(x - np.max(x)) # Subtract max for numerical stability
    return e_x / e_x.sum(axis=0)

# 3. Initialize Weights
embedding_dim = 3
learning_rate = 0.05
epochs = 1000

# W1: Hidden layer (This becomes our actual Word Embeddings!)
W1 = np.random.randn(V, embedding_dim) 
# W2: Output layer
W2 = np.random.randn(embedding_dim, V)

# 4. Training Loop (Stochastic Gradient Descent)
for epoch in range(epochs):
    loss = 0
    for center_id, context_id in training_data:
        # Forward Pass
        x = one_hot(center_id, V).reshape(-1, 1) # Input vector [V x 1]
        h = np.dot(W1.T, x)                      # Hidden layer [dim x 1]
        u = np.dot(W2.T, h)                      # Output scores [V x 1]
        y_pred = softmax(u)                      # Probabilities [V x 1]
        
        # Calculate Loss (Cross Entropy)
        loss -= np.log(y_pred[context_id][0])
        
        # Backward Pass (Gradients)
        e = y_pred.copy()
        e[context_id] -= 1.0                     # Error: (y_hat - y)
        
        dW2 = np.dot(h, e.T)                     # Gradient for W2
        dW1 = np.dot(x, np.dot(W2, e).T)         # Gradient for W1
        
        # Update Weights
        W2 -= learning_rate * dW2
        W1 -= learning_rate * dW1

print("Training Complete. Final Loss:", loss)

# Retrieve the embedding for "king"
print(f"Embedding for 'king': {W1[word2id['king']]}")

Vocabulary Size: 10
Center Word: the, Context Words: ['king']
Training Data: [(3, 0)]
Center Word: king, Context Words: ['the', 'is']
Training Data: [(3, 0), (0, 3), (0, 4)]
Center Word: is, Context Words: ['king', 'a']
Training Data: [(3, 0), (0, 3), (0, 4), (4, 0), (4, 7)]
Center Word: a, Context Words: ['is', 'strong']
Training Data: [(3, 0), (0, 3), (0, 4), (4, 0), (4, 7), (7, 4), (7, 8)]
Center Word: strong, Context Words: ['a', 'man']
Training Data: [(3, 0), (0, 3), (0, 4), (4, 0), (4, 7), (7, 4), (7, 8), (8, 7), (8, 2)]
Center Word: man, Context Words: ['strong', 'and']
Training Data: [(3, 0), (0, 3), (0, 4), (4, 0), (4, 7), (7, 4), (7, 8), (8, 7), (8, 2), (2, 8), (2, 6)]
Center Word: and, Context Words: ['man', 'the']
Training Data: [(3, 0), (0, 3), (0, 4), (4, 0), (4, 7), (7, 4), (7, 8), (8, 7), (8, 2), (2, 8), (2, 6), (6, 2), (6, 3)]
Center Word: the, Context Words: ['and', 'queen']
Training Data: [(3, 0), (0, 3), (0, 4), (4, 0), (4, 7), (7, 4), (7, 8), (8, 7), (8, 2), (2, 8)

### Evaluating Our Custom Word Embeddings
Now that we have trained our `W1` weight matrix, we have effectively generated our word embeddings! Each row in `W1` corresponds to the dense vector representation of a word in our vocabulary.

A fascinating property of Word2Vec is that these vectors capture semantic meaning. Words that appear in similar contexts will have vectors pointing in similar directions. We can measure this using **Cosine Similarity**. 

Furthermore, we can perform **Vector Arithmetic** (e.g., subtracting vectors) to see how words relate to each other in this geometric space.

*Note: Real-world embeddings are trained on billions of words. Because our toy corpus is extremely small, our vectors won't perfectly capture complex analogies, but the mathematical pipeline remains exactly the same!*

In [11]:
def calculate_cosine_similarity(vec_a, vec_b):
    """Calculates the cosine similarity between two vectors."""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    # Avoid division by zero
    if norm_a == 0 or norm_b == 0:
        return 0.0
        
    return dot_product / (norm_a * norm_b)

def find_closest_word(target_vec, word2id, id2word, W1):
    """Finds the word in the vocabulary with the highest cosine similarity to a target vector."""
    best_word = None
    highest_sim = -float('inf')
    
    for word, idx in word2id.items():
        word_vec = W1[idx]
        sim = calculate_cosine_similarity(target_vec, word_vec)
        
        if sim > highest_sim:
            highest_sim = sim
            best_word = word
            
    return best_word, highest_sim

# 1. Extract Vectors for Specific Words
vec_king = W1[word2id['king']]
vec_man = W1[word2id['man']]
vec_queen = W1[word2id['queen']]
vec_woman = W1[word2id['woman']]

# 2. Check Cosine Similarity
print("--- Cosine Similarities ---")
sim_king_man = calculate_cosine_similarity(vec_king, vec_man)
sim_king_queen = calculate_cosine_similarity(vec_king, vec_queen)

print(f"Similarity (King, Man): {sim_king_man:.4f}")
print(f"Similarity (King, Queen): {sim_king_queen:.4f}")

# 3. Vector Arithmetic (Analogy: King - Man + Woman)
print("\n--- Vector Arithmetic ---")
print("Target: King - Man + Woman")

# Perform the subtraction and addition
target_vector = vec_king - vec_man + vec_woman

# Find the closest word in our vocabulary to this new synthetic vector
closest_word, match_score = find_closest_word(target_vector, word2id, id2word, W1)

print(f"Predicted closest word: '{closest_word}' (Similarity: {match_score:.4f})")
print(f"Actual vector for '{closest_word}':\n{W1[word2id[closest_word]]}")

--- Cosine Similarities ---
Similarity (King, Man): -0.3623
Similarity (King, Queen): 0.9992

--- Vector Arithmetic ---
Target: King - Man + Woman
Predicted closest word: 'woman' (Similarity: 0.7126)
Actual vector for 'woman':
[ 3.32884974 -2.53528264 -0.35954293]


---
## Topic 2: Naive Bayes Classifier From Scratch

Naive Bayes is a probabilistic classifier based on Bayes' Theorem. It makes the "naive" assumption that features (words) are independent of each other given the class label.

To classify a document $D$ into a class $c$, we calculate the posterior probability:

$$P(c \mid D) \propto P(c) \prod_{i=1}^{n} P(w_i \mid c)$$

To prevent zero probabilities for unseen words, we use **Laplace (Add-1) Smoothing**:

$$P(w_i \mid c) = \frac{\text{count}(w_i, c) + 1}{\sum_{w \in V} \text{count}(w, c) + |V|}$$

*Note: We use log probabilities in the code to prevent numerical underflow when multiplying many small fractions.*

In [12]:
# Toy Sentiment Dataset
docs = [
    ("I love this amazing movie", "positive"),
    ("what a great and fantastic film", "positive"),
    ("I hate this terrible movie", "negative"),
    ("what a bad and boring film", "negative")
]

def train_naive_bayes(data):
    classes = defaultdict(int)
    word_counts = defaultdict(lambda: defaultdict(int))
    vocab = set()
    total_docs = len(data)
    
    # Count frequencies
    for text, label in data:
        classes[label] += 1
        words = text.lower().split()
        for word in words:
            word_counts[label][word] += 1
            vocab.add(word)
            
    # Calculate Log Priors and Log Likelihoods
    log_priors = {}
    log_likelihoods = defaultdict(dict)
    V_size = len(vocab)
    
    for c in classes:
        # Prior: P(c)
        log_priors[c] = np.log(classes[c] / total_docs)
        
        # Total words in this class
        total_words_in_c = sum(word_counts[c].values())
        
        # Likelihoods with Laplace Smoothing: P(w | c)
        for word in vocab:
            count = word_counts[c][word]
            prob = (count + 1) / (total_words_in_c + V_size)
            log_likelihoods[c][word] = np.log(prob)
            
    return log_priors, log_likelihoods, vocab

# Train the model
log_priors, log_likelihoods, model_vocab = train_naive_bayes(docs)

def predict_naive_bayes(text, priors, likelihoods, vocab):
    words = text.lower().split()
    scores = {}
    
    for c in priors:
        # Start with the log prior
        scores[c] = priors[c]
        for word in words:
            if word in vocab:
                # Add the log likelihood
                scores[c] += likelihoods[c][word]
                
    # Return the class with the highest score
    return max(scores, key=scores.get), scores

# Test the model
test_doc = "this movie is great"
pred_class, scores = predict_naive_bayes(test_doc, log_priors, log_likelihoods, model_vocab)

print(f"Text: '{test_doc}'")
print(f"Predicted Class: {pred_class}")
print(f"Log Probability Scores: {scores}")

Text: 'this movie is great'
Predicted Class: positive
Log Probability Scores: {'positive': np.float64(-8.387995252944556), 'negative': np.float64(-9.081142433504501)}


---
## Topic 3: Logistic Regression Classifier From Scratch

Logistic Regression is a discriminative model. Instead of modeling the joint probability like Naive Bayes, it directly maps features (like Bag of Words vectors) to probabilities using the **Sigmoid** function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
Where $z = \mathbf{X}W + b$.

We train the model by calculating the gradient of the Log Loss and updating our weights $W$ and bias $b$:
$$W := W - \alpha \frac{1}{m} \mathbf{X}^T(\hat{y} - y)$$

In [13]:
# Create a Bag of Words matrix for our toy dataset
def create_bow_matrix(docs, vocab):
    vocab_list = list(vocab)
    X = np.zeros((len(docs), len(vocab_list)))
    y = np.zeros(len(docs))
    
    for i, (text, label) in enumerate(docs):
        y[i] = 1 if label == "positive" else 0
        words = text.lower().split()
        for word in words:
            if word in vocab_list:
                X[i, vocab_list.index(word)] += 1
    return X, y, vocab_list

X_train, y_train, vocab_list = create_bow_matrix(docs, model_vocab)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_logistic_regression(X, y, learning_rate=0.1, epochs=1000):
    m, n = X.shape
    weights = np.zeros(n)
    bias = 0.0
    
    for epoch in range(epochs):
        # Forward pass: Compute predictions
        linear_model = np.dot(X, weights) + bias
        y_pred = sigmoid(linear_model)
        
        # Calculate Gradients
        dw = (1 / m) * np.dot(X.T, (y_pred - y))
        db = (1 / m) * np.sum(y_pred - y)
        
        # Update parameters
        weights -= learning_rate * dw
        bias -= learning_rate * db
        
    return weights, bias

# Train the model
weights, bias = train_logistic_regression(X_train, y_train)

def predict_logistic_regression(text, weights, bias, vocab_list):
    # Convert text to BoW vector
    x_test = np.zeros(len(vocab_list))
    for word in text.lower().split():
        if word in vocab_list:
            x_test[vocab_list.index(word)] += 1
            
    # Apply model
    linear_model = np.dot(x_test, weights) + bias
    prob = sigmoid(linear_model)
    
    pred_class = "positive" if prob >= 0.5 else "negative"
    return pred_class, prob

# Test the model
test_text = "amazing and fantastic"
lr_pred, lr_prob = predict_logistic_regression(test_text, weights, bias, vocab_list)

print(f"Text: '{test_text}'")
print(f"Predicted Class: {lr_pred} (Probability of Positive: {lr_prob:.4f})")

Text: 'amazing and fantastic'
Predicted Class: positive (Probability of Positive: 0.9793)


---
## Topic 4: Using Word2Vec for Logistic Regression
Instead of using sparse Bag of Words vectors, we can use our trained Word2Vec embeddings to classify text. 

To convert a multi-word sentence into a single mathematical vector that Logistic Regression can process, we take the **Mean Word Embedding**—simply averaging the Word2Vec vectors for every valid word in the sentence:

$$X_{\text{doc}} = \frac{1}{N} \sum_{i=1}^{N} \vec{w}_i$$

In [15]:
def get_document_embedding(text, word2id, W1, embedding_dim):
    """
    Converts a sentence into a single dense vector by averaging its word embeddings.
    """
    words = text.lower().split()
    valid_vectors = []
    
    for word in words:
        if word in word2id:
            valid_vectors.append(W1[word2id[word]])
            
    # If no words in the text are in our vocabulary, return a zero vector
    if len(valid_vectors) == 0:
        return np.zeros(embedding_dim)
        
    # Calculate the mean across the column axis (axis=0)
    doc_embedding = np.mean(valid_vectors, axis=0)
    return doc_embedding

# 1. Convert our toy dataset into dense Word2Vec document representations
embedding_dim = W1.shape[1]
X_train_dense = np.zeros((len(docs), embedding_dim))
y_train_dense = np.zeros(len(docs))

for i, (text, label) in enumerate(docs):
    y_train_dense[i] = 1 if label == "positive" else 0
    X_train_dense[i] = get_document_embedding(text, word2id, W1, embedding_dim)

# 2. Train Logistic Regression on the Dense Vectors
# (We reuse the train_logistic_regression function from earlier!)
weights_dense, bias_dense = train_logistic_regression(X_train_dense, y_train_dense, learning_rate=0.1, epochs=1000)

# 3. Test the Dense Classifier
test_text = "amazing and fantastic"
test_doc_vector = get_document_embedding(test_text, word2id, W1, embedding_dim)

# Apply the linear model and sigmoid function
linear_z = np.dot(test_doc_vector, weights_dense) + bias_dense
prob_dense = sigmoid(linear_z)
pred_dense = "positive" if prob_dense >= 0.5 else "negative"

print(f"Text: '{test_text}'")
print(f"Predicted Class (Word2Vec + LR): {pred_dense} (Probability: {prob_dense:.4f})")

Text: 'amazing and fantastic'
Predicted Class (Word2Vec + LR): positive (Probability: 0.5000)


## Topic 5: Logistic Regression with TF-IDF Weighted Embeddings

While taking the simple mean of Word2Vec embeddings is a decent baseline, it dilutes important words. We can upgrade this by multiplying each word's embedding by its TF-IDF score before summing them.

In this section, we will:
1. Compute the IDF dictionary for our corpus.
2. Convert our documents into TF-IDF weighted Word2Vec representations.
3. Train our custom Logistic Regression classifier on these new dense vectors.
4. Test the famous word-order limitation: "The dog bit the man" vs. "The man bit the dog".

In [17]:
import numpy as np
from collections import Counter

# 1. Setup Toy Data and Mock Word2Vec
docs = [
    ("I love this amazing movie", "positive"),
    ("what a great and fantastic film", "positive"),
    ("I hate this terrible movie", "negative"),
    ("what a bad and boring film", "negative")
]

# Build vocabulary
vocab = list(set(" ".join([text for text, label in docs]).lower().split() + ["the", "dog", "bit", "man"]))
word2id = {w: i for i, w in enumerate(vocab)}
embedding_dim = 5
W1 = np.random.randn(len(vocab), embedding_dim) # Mock Word2Vec matrix

# 2. Compute IDF Dictionary
def compute_idf(data, vocab_list):
    N = len(data)
    idf_dict = {}
    for word in vocab_list:
        # Count how many documents contain the word
        df = sum(1 for text, _ in data if word in text.lower().split())
        # Add 1 smoothing to prevent division by zero
        idf_dict[word] = np.log((N + 1) / (df + 1)) + 1
    return idf_dict

idf_lookup = compute_idf(docs, vocab)

# 3. TF-IDF Weighted Embedding Function
def get_tfidf_embedding(text, word2id_map, word_matrix, idf_dict, dim):
    words = text.lower().split()
    valid_words = [w for w in words if w in word2id_map]
    
    if not valid_words:
        return np.zeros(dim)
        
    tf_counts = Counter(valid_words)
    total_words = len(valid_words)
    
    weighted_sum = np.zeros(dim)
    total_weight = 0.0
    
    for word in valid_words:
        tf = tf_counts[word] / total_words
        idf = idf_dict.get(word, 1.0) # Default to 1.0 if unseen
        weight = tf * idf
        
        weighted_sum += word_matrix[word2id_map[word]] * weight
        total_weight += weight
        
    return weighted_sum / total_weight if total_weight > 0 else np.zeros(dim)

def get_mean_embedding(text, word2id_map, word_matrix, dim):
    words = text.lower().split()
    valid_words = [w for w in words if w in word2id_map]
    if not valid_words:
        return np.zeros(dim)
    vectors = [word_matrix[word2id_map[w]] for w in valid_words]
    return np.mean(vectors, axis=0)

# 4. Prepare Training Data for Logistic Regression
X_train_tfidf = np.zeros((len(docs), embedding_dim))
y_train = np.zeros(len(docs))

for i, (text, label) in enumerate(docs):
    y_train[i] = 1 if label == "positive" else 0
    X_train_tfidf[i] = get_tfidf_embedding(text, word2id, W1, idf_lookup, embedding_dim)

# 5. Train Logistic Regression
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -250, 250)))

def train_lr(X, y, lr=0.1, epochs=1000):
    m, n = X.shape
    weights = np.zeros(n)
    bias = 0.0
    
    for _ in range(epochs):
        y_pred = sigmoid(np.dot(X, weights) + bias)
        dw = (1 / m) * np.dot(X.T, (y_pred - y))
        db = (1 / m) * np.sum(y_pred - y)
        weights -= lr * dw
        bias -= lr * db
    return weights, bias

weights_tfidf, bias_tfidf = train_lr(X_train_tfidf, y_train)
print("Model trained successfully on TF-IDF weighted embeddings!\n")

Model trained successfully on TF-IDF weighted embeddings!



### The Word Order Test: Mean vs. TF-IDF

Now let's test if TF-IDF weighting solves the anagram problem. We will generate both the Mean Average vectors and the TF-IDF Weighted vectors for:
* **Sentence A:** "the dog bit the man"
* **Sentence B:** "the man bit the dog"

In [19]:
sentence_a = "the dog bit the man"
sentence_b = "the man bit the dog"

# Test Mean Average
mean_vec_a = get_mean_embedding(sentence_a, word2id, W1, embedding_dim)
mean_vec_b = get_mean_embedding(sentence_b, word2id, W1, embedding_dim)

# Test TF-IDF Weighted Average
tfidf_vec_a = get_tfidf_embedding(sentence_a, word2id, W1, idf_lookup, embedding_dim)
tfidf_vec_b = get_tfidf_embedding(sentence_b, word2id, W1, idf_lookup, embedding_dim)

print("--- MEAN AVERAGE TEST ---")
print(f"Are Mean vectors identical? {np.allclose(mean_vec_a, mean_vec_b)}")

print("\n--- TF-IDF WEIGHTED TEST ---")
print(f"Are TF-IDF vectors identical? {np.allclose(tfidf_vec_a, tfidf_vec_b)}")

print("\nExplanation:")
print("Because both sentences have the exact same vocabulary counts (2 'the', 1 'dog', 1 'bit', 1 'man'),")
print("their TF scores and IDF scores are identical. Any pure 'Bag of Words' aggregation method,")
print("including TF-IDF weighting, fundamentally ignores sequence and will produce identical document vectors.")
print("To solve this, we must upgrade to Recurrent Neural Networks (RNNs)!")

--- MEAN AVERAGE TEST ---
Are Mean vectors identical? True

--- TF-IDF WEIGHTED TEST ---
Are TF-IDF vectors identical? True

Explanation:
Because both sentences have the exact same vocabulary counts (2 'the', 1 'dog', 1 'bit', 1 'man'),
their TF scores and IDF scores are identical. Any pure 'Bag of Words' aggregation method,
including TF-IDF weighting, fundamentally ignores sequence and will produce identical document vectors.
To solve this, we must upgrade to Recurrent Neural Networks (RNNs)!
